In [0]:
# Silver Layer - Dimensions Notebook
# Handles SCD Type 1 (Users, Tracks, Artists, Albums) and Date Dimension

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import datetime

# Set checkpoint directory (Unity Catalog compatible)
spark.conf.set("spark.sql.streaming.checkpointLocation", "/tmp/silver_checkpoints")

catalog = "spotify_catalog"
bronze_db = "bronze"
silver_db = "silver"

print("Silver Dimensions notebook initialized")

In [0]:
# ============================================================
# CELL 2: dim_users - SCD Type 1 (Overwrite on change)
# ============================================================

def process_dim_users():
    bronze_users = spark.table(f"{catalog}.{bronze_db}.users")
    
    df = bronze_users.select(
        F.col("user_id"),
        F.col("user_name"),
        F.col("email"),
        F.col("country"),
        F.col("subscription_type"),
        F.col("age").cast(IntegerType()),
        F.col("gender"),
        F.to_date(F.col("registration_date")).alias("registration_date"),
        F.current_timestamp().alias("updated_at")
    ).dropDuplicates(["user_id"])
    
    target_table = f"{catalog}.{silver_db}.dim_users"
    
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        dt.alias("target").merge(
            df.alias("source"),
            "target.user_id = source.user_id"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print(f"dim_users: MERGE complete")
    else:
        df.write.format("delta").mode("overwrite") \
            .saveAsTable(target_table)
        print(f"dim_users: Created new table")
    
    return spark.table(target_table)

df_users = process_dim_users()
display(df_users)

In [0]:
# ============================================================
# CELL 3: dim_tracks - SCD Type 1
# ============================================================

def process_dim_tracks():
    bronze_tracks = spark.table(f"{catalog}.{bronze_db}.tracks")
    
    df = bronze_tracks.select(
        F.col("track_id"),
        F.col("track_name"),
        F.col("artist_id"),
        F.col("album_id"),
        F.col("duration_ms").cast(LongType()),
        F.col("explicit").cast(BooleanType()),
        F.col("popularity").cast(IntegerType()),
        F.col("genre"),
        F.current_timestamp().alias("updated_at")
    ).dropDuplicates(["track_id"])
    
    target_table = f"{catalog}.{silver_db}.dim_tracks"
    
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        dt.alias("target").merge(
            df.alias("source"),
            "target.track_id = source.track_id"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print("dim_tracks: MERGE complete")
    else:
        df.write.format("delta").mode("overwrite") \
            .saveAsTable(target_table)
        print("dim_tracks: Created new table")
    
    return spark.table(target_table)

df_tracks = process_dim_tracks()
display(df_tracks)

In [0]:
# ============================================================
# CELL 4: dim_artists - SCD Type 1
# ============================================================

def process_dim_artists():
    bronze_artists = spark.table(f"{catalog}.{bronze_db}.artists")
    
    df = bronze_artists.select(
        F.col("artist_id"),
        F.col("artist_name"),
        F.col("genres"),
        F.col("followers").cast(LongType()),
        F.col("popularity").cast(IntegerType()),
        F.current_timestamp().alias("updated_at")
    ).dropDuplicates(["artist_id"])
    
    target_table = f"{catalog}.{silver_db}.dim_artists"
    
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        dt.alias("target").merge(
            df.alias("source"),
            "target.artist_id = source.artist_id"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print("dim_artists: MERGE complete")
    else:
        df.write.format("delta").mode("overwrite") \
            .saveAsTable(target_table)
        print("dim_artists: Created new table")
    
    return spark.table(target_table)

df_artists = process_dim_artists()
display(df_artists)

In [0]:
# ============================================================
# CELL 5: dim_albums - SCD Type 1
# ============================================================

def process_dim_albums():
    bronze_albums = spark.table(f"{catalog}.{bronze_db}.albums")
    
    df = bronze_albums.select(
        F.col("album_id"),
        F.col("album_name"),
        F.col("artist_id"),
        F.col("album_type"),
        F.col("total_tracks").cast(IntegerType()),
        F.to_date(F.col("release_date")).alias("release_date"),
        F.col("label"),
        F.col("popularity").cast(IntegerType()),
        F.current_timestamp().alias("updated_at")
    ).dropDuplicates(["album_id"])
    
    target_table = f"{catalog}.{silver_db}.dim_albums"
    
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        dt.alias("target").merge(
            df.alias("source"),
            "target.album_id = source.album_id"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print("dim_albums: MERGE complete")
    else:
        df.write.format("delta").mode("overwrite") \
            .saveAsTable(target_table)
        print("dim_albums: Created new table")
    
    return spark.table(target_table)

df_albums = process_dim_albums()
display(df_albums)

In [0]:
# ============================================================
# CELL 6: dim_date - Date Dimension (SCD Type 0 - Static)
# ============================================================

def process_dim_date(start_year=2015, end_year=2030):
    target_table = f"{catalog}.{silver_db}.dim_date"
    
    if spark.catalog.tableExists(target_table):
        print("dim_date: Already exists, skipping")
        return spark.table(target_table)
    
    start_date = datetime.date(start_year, 1, 1)
    end_date = datetime.date(end_year, 12, 31)
    
    date_list = []
    current = start_date
    while current <= end_date:
        date_list.append((
            int(current.strftime("%Y%m%d")),   # date_key
            current,                            # full_date
            current.year,                       # year
            current.month,                      # month
            current.day,                        # day
            current.isocalendar()[1],           # week_of_year
            current.weekday() + 1,             # day_of_week (1=Mon)
            current.strftime("%B"),             # month_name
            current.strftime("%A"),             # day_name
            current.weekday() >= 5,            # is_weekend
            (current.month - 1) // 3 + 1       # quarter
        ))
        current += datetime.timedelta(days=1)
    
    schema = StructType([
        StructField("date_key", IntegerType(), False),
        StructField("full_date", DateType(), False),
        StructField("year", IntegerType(), False),
        StructField("month", IntegerType(), False),
        StructField("day", IntegerType(), False),
        StructField("week_of_year", IntegerType(), False),
        StructField("day_of_week", IntegerType(), False),
        StructField("month_name", StringType(), False),
        StructField("day_name", StringType(), False),
        StructField("is_weekend", BooleanType(), False),
        StructField("quarter", IntegerType(), False)
    ])
    
    df_date = spark.createDataFrame(date_list, schema=schema)
    df_date.write.format("delta").mode("overwrite") \
        .saveAsTable(target_table)
    print(f"dim_date: Created with {df_date.count()} rows")
    return spark.table(target_table)

df_date = process_dim_date()
display(df_date)

In [0]:
# Ensure catalog and schema exist
spark.sql("CREATE CATALOG IF NOT EXISTS spotify_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS spotify_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS spotify_catalog.bronze")

# Auto Loader: Stream users from Bronze ADLS -> Silver Delta table (batch trigger)
df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimUser/checkpoint") \
    .load("abfss://bronze@marufazureproject.dfs.core.windows.net/DimUser")

query = df_user.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimUser/checkpoint") \
    .option("mergeSchema", "true") \
    .toTable("spotify_catalog.silver.dim_users")

query.awaitTermination()
print("Streaming batch complete")

In [0]:
# Preview the silver dim_users table
display(spark.table("spotify_catalog.silver.dim_users"))

In [0]:
# ============================================================
# CELL 9: Auto Loader - Stream dim_tracks Bronze -> Silver
# ============================================================

df_tracks = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimTrack/checkpoint") \
    .load("abfss://bronze@marufazureproject.dfs.core.windows.net/DimTrack")

query_tracks = df_tracks.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimTrack/checkpoint") \
    .option("mergeSchema", "true") \
    .toTable("spotify_catalog.silver.dim_tracks")

query_tracks.awaitTermination()
print("dim_tracks stream complete")

In [0]:
# ============================================================
# CELL 11: Auto Loader - Stream dim_albums Bronze -> Silver
# ============================================================

df_albums = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimAlbum/checkpoint") \
    .load("abfss://bronze@marufazureproject.dfs.core.windows.net/DimAlbum")

query_albums = df_albums.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimAlbum/checkpoint") \
    .option("mergeSchema", "true") \
    .toTable("spotify_catalog.silver.dim_albums")

query_albums.awaitTermination()
print("dim_albums stream complete")

In [0]:
# ============================================================
# CELL 12: Auto Loader - Stream fact_streams Bronze -> Silver
# ============================================================

df_streams = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/FactStream/checkpoint") \
    .load("abfss://bronze@marufazureproject.dfs.core.windows.net/StreamingHistory")

query_streams = df_streams.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/FactStream/checkpoint") \
    .option("mergeSchema", "true") \
    .toTable("spotify_catalog.silver.fact_streams")

query_streams.awaitTermination()
print("fact_streams stream complete")